In [1]:
import pandas as pd
import numpy as np

PROJECT = '../data/processed'

pod_wide = pd.read_pickle(f'{PROJECT}/podlesny_wide.pkl')
pod_meta = pd.read_csv(f'{PROJECT}/podlesny_meta_split.csv')

recipient_types = ['Post', 'Pre No-ABx', 'Pre Post-ABx']
recipients = pod_meta[pod_meta['Sample_Type'].isin(recipient_types)]

print('Recipient samples per disease group:')
print(recipients['Disease_Group'].value_counts())

def preprocess(df, group_name, prevalence_thresh=0.20):
    print(f"\n{group_name}")
    print("Start:", df.shape)

    if df.values.max() > 1:
        df = df / 100.0
        print("  Input detected as percentage scale — rescaled by /100")
    else:
        print("  Input detected as proportion scale — no rescaling applied")

    df = df.loc[:, df.sum(axis=0) > 0]

    prevalence = (df > 0).mean(axis=0)
    df = df.loc[:, prevalence >= prevalence_thresh]
    print("After prevalence:", df.shape)

    n_before = df.shape[0]
    df = df.loc[df.sum(axis=1) > 0]
    n_dropped = n_before - df.shape[0]
    if n_dropped > 0:
        print(f"  Dropped {n_dropped} all-zero sample(s)")

    df_norm = df.div(df.sum(axis=1), axis=0)

    df_pseudo = df_norm + 1e-6
    log_df = np.log(df_pseudo)
    df_clr = log_df.sub(log_df.mean(axis=1), axis=0)

    df_counts = (df_norm * 10000).round().astype(int)

    prevalence = (df_counts > 0).sum(axis=0) / df_counts.shape[0]
    df_counts = df_counts.loc[:, prevalence > 0.05]

    keep = df_counts.columns
    df_norm = df_norm[keep]
    df_clr = df_clr[keep]

    return df_norm, df_clr, df_counts

GROUPS = ['IBD', 'rCDI', 'MetS', 'MDR', 'ICI']

for grp in GROUPS:
    sample_ids = recipients.loc[recipients['Disease_Group'] == grp, 'Name']
    sample_ids = [s for s in sample_ids if s in pod_wide.index]
    df_grp = pod_wide.loc[sample_ids]

    norm, clr, counts = preprocess(df_grp, grp)

    norm.to_csv(f'{PROJECT}/{grp}_pod_network.csv')
    clr.to_csv(f'{PROJECT}/{grp}_pod_network_CLR.csv')
    counts.to_csv(f'{PROJECT}/{grp}_pod_network_counts.tsv', sep='\t')

    print(f'{grp}: saved {counts.shape[0]} samples x {counts.shape[1]} species\n')

Recipient samples per disease group:
Disease_Group
IBD     353
MetS    317
ICI     220
rCDI    127
MDR     102
Name: count, dtype: int64

IBD
Start: (353, 860)
  Input detected as proportion scale — no rescaling applied
After prevalence: (353, 95)
IBD: saved 353 samples x 95 species


rCDI
Start: (127, 860)
  Input detected as proportion scale — no rescaling applied
After prevalence: (127, 158)
rCDI: saved 127 samples x 158 species


MetS
Start: (317, 860)
  Input detected as proportion scale — no rescaling applied
After prevalence: (317, 180)
MetS: saved 317 samples x 179 species


MDR
Start: (102, 860)
  Input detected as proportion scale — no rescaling applied
After prevalence: (102, 111)
MDR: saved 102 samples x 111 species


ICI
Start: (220, 860)
  Input detected as proportion scale — no rescaling applied
After prevalence: (220, 115)
ICI: saved 220 samples x 115 species

